
<div style="background-color:#EFF5FF; padding:14px; border-radius:10px; border-left:8px solid #009E91;">

<h2 style="margin-bottom:0;"><b>Text Mining - Final Project</b></h2>
<h2 style="margin-bottom:0;"><b>Notebook 2: Data Exploration and Preprocessing</b></h2>


<h3 style="margin-top:5px; color:#009E91;;"><b>"Decoding the Rhythms of Emotion: A Sentimental Journey Through Music Genres" </b></h3>

### **1. Imports and Data Loading**

#### **1.1. Imports and Libraries**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re

#### **1.2. Loading the Data**

For this project, the data is divided into two subsets:
* **Train Dataset ( `train.csv` ):** It contains the song features and our target variable (`tag`, which is the musical genre). We will use this set to build and train our machine-learning models.
* **Test Dataset (`test.csv`):** It also contains the same features apart from `tag`. We will use our final model to predict the genres for this unseen data.

**Train Dataset**

In [ ]:
train_dataset = pd.read_csv('./Datasets/train.csv', index_col = 'id')
train_dataset.head()

**Test Dataset**

In [ ]:
test_dataset = pd.read_csv('./Datasets/test.csv', index_col = 'id')
test_dataset.head()

### **2. Initial Overview**

##### **2.1. Features and Target Variable**

Before moving to the exploratory analysis of our data, it is important to understand what are the variables we will be working with about. The datasets contain the following features:

**Features**
* **`id`**: Unique identifier (currently set as our dataframe index)
* **`title`:** Song title
* **`artist`:** The main artist of the song.
* **`year`:** Year the song was launched in.
* **`views`:** Number of views the song has on an online music service.
* **`features`:** Includes names of collaborating artists, empty otherwise.
* **`lyrics`:** The lyrics of the song. Includes tags for specific sections (e.g. Introduction, Chorus).

**Target Variable**
* **`tag`**: The main musical genre of the song. (This is what our classification model will attempt to predict)

**Unnecessary Columns for Modelling**

We noticed that most of the features we have in our datasets **aren't useful for modelling purposes**. For instance, leaving the `artist` column would make the model 'memorize' that a certain artist is associated with a specific genre (e.g. Eminem = rap), meaning the model would not learn anything about the lyrics themselves. Leaving `views` and `year` also doesn't make sense as they are numerical and do not contribute to the lyrics analysis.

However, while these features won't be used for the final model, **we will keep them in the dataset** rather than dropping them. They are essential to answer some business questions and in the Sentimental Analysis phase.

In [ ]:
train_dataset = train_dataset.rename(columns={'lyrics': 'raw_lyrics'})
test_dataset = test_dataset.rename(columns={'lyrics': 'raw_lyrics'}) 

##### **2.2. Dataset Dimensions, Data types and Missing Values**

In this section, we'll check the dimensions of both datasets, data types, and look for missing records to ensure our datasets are reliable and also to identify any immediate cleaning requirements.

In [ ]:
# Check the shape of the datasets
print(f"Train dataset shape: {train_dataset.shape}")
print(f"Test dataset shape: {test_dataset.shape}")

In [ ]:
# Display data types and non-null counts for both datasets
print("\n--- Train Dataset Info ---")
print('**************************')

train_dataset.info()

print("\n--- Test Dataset Info ---")
print('*************************')

test_dataset.info()

**Observations:**

The training dataset contains 134,967 rows and 7 columns, while the test dataset consists of 33,742 rows and 6 columns (which was expected due to ther target variable).

The `.info()` summary indicates that our features are a mix of numerical (`int64`) and categorical (`object`) formats. Additionally, there are **no significant missing values** across the datasets.

Although the `raw_lyrics` column is initially loaded as an `object`, we decided to transform it into a string format so we can prevent future error due to the regex operations.

In [ ]:
#change the data type of the 'raw_lyrics' column to string for both datasets, so we can make regex operations.
train_dataset['raw_lyrics'] = train_dataset['raw_lyrics'].astype(str)
test_dataset['raw_lyrics'] = test_dataset['raw_lyrics'].astype(str) 

##### **2.3. Statistical Summary**

In [ ]:
train_dataset.describe(include='all')

**Observations:**

*   **Target Variable (`tag`):** There are 6 unique musical genres, with "pop" being the most dominant class (55,742 occurrences). Our next step should be to verify the target variable distribution.
*   **Year Column:** The `year` feature contains obvious entry errors, such as a minimum value of `1`. 
*   **Lyrics:** Out of 134,967 rows, the most frequent entry in `raw_lyrics` is the following string: "Tell us that you would like to have the lyrics...". We will try to verify the existence of duplicate records, if that is the case.
*   **Numerical Distributions:** The `views` column has a massive range (from 0 to over 3.6 million) and a very high standard deviation.
*   **Collaborations:** The majority of the songs do not have featured artists, as the most frequent value in the `features` column is an empty bracket `{}`  with 111,928 occurrences.

##### **2.4.Duplicated Records**

In [ ]:
print(f"Train dataset duplicated rows: {train_dataset.duplicated().sum()}")
print(f"Test dataset duplicated rows: {test_dataset.duplicated().sum()}")

**Observations**
We verify that there are no exact duplicates across all columns in our dataset. However, a close look at our statistical summary revealed that only 134,789 out of 134,967 total entries are unique, this means that there are identical lyrics assigned to different tracks.

In [ ]:
lyrics_duplicates = train_dataset['raw_lyrics'].duplicated().sum() #duplicated lyrics in the train dataset 
print(f"Number of duplicated lyrics: {lyrics_duplicates}\n")


top_repeated = train_dataset['raw_lyrics'].value_counts().head(3)
top_lyric_text = top_repeated.index[0]

example = train_dataset[train_dataset['raw_lyrics'] == top_lyric_text]
print(example[['artist', 'title', 'raw_lyrics']].head(3))

In [ ]:
print(example['raw_lyrics'].iloc[0])

In [ ]:
# Remove the dynamically identified scraping placeholder from the training set
train_dataset = train_dataset[train_dataset['raw_lyrics'] != top_lyric_text]

print(f"Train dataset shape after dropping placeholders: {train_dataset.shape}")

## **ANALISAR ESTE DESCRIBE**
O ano tem minimo de 1, ha letras que sao exatamente iguais, ver o que se passa uma vez que nao ha duplicados

### **3. Exploratory Data Analysis (EDA) and Preprocessing**

#### **3.1. Target Variable Distribution**

It is important to analyse the distribution of our target variable ( `tag` ). Having a heavily imbalanced dataset, which, in this case, means having significantly more songs in some genres than the others, could bias our machine learning model toward the majority classes. The model might try to minimize its error rate failing to properly learn and identify the minority class.

In [ ]:
plt.figure(figsize=(12, 4))
sns.countplot(
    data=train_dataset, 
    x='tag', 
    color='skyblue', 
    order=train_dataset['tag'].value_counts().index)

plt.title('Distribution of Tags in Train Dataset', fontsize=12, fontweight='bold')
plt.xlabel('')
plt.ylabel('')

plt.tight_layout()
plt.show()

##### **Observations:**

As we can observe on the plot above, our dataset is clearly imbalanced being the majority class `pop`and the minority `country`. This is an useful insight because it means that solely relying on '**Accuracy**' as a performance metric could be **misleading**.

#### **3.2. Lyrics Length Analysis**

In this section, we'll analyse with details the **length of the lyrics in our dataset**. Typically, a song ranges between 150 and 600 words. High words count could indicate the presence of book chapters, theatre scripts, blogs etc. We'll begin by calculating the **word count for each song**.

In [ ]:
train_dataset['word_count'] = train_dataset['raw_lyrics'].apply(lambda x: len(str(x).split())) #Function to count the number of words in the lyrics

print(train_dataset['word_count'].describe()) #Statistical summary of the word count in the lyrics

**Observations:**


The stats above reveal a massive disparity between the lyrics' length. While the average is around ~320 words, which was expected already, the maximum value reaches 18221 words. The presence of this entry plus the high standard deviation (~451), confirms us the presence of several outliers. 

##### **3.2.1 Distribution of Word Counts**

To understand the distribution of the lengths and identify potential outliers, we **plot the word counts using both a linear and a logarithmic scale**. The linear scale helps us observe where the vast majority of our data is concentrated, while the logarithmic scale allows us to examine the "long tail" of extreme values that would otherwise remain hidden.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

#Graph 1: Distribution in Linear Scale
sns.histplot(train_dataset['word_count'], bins=100, color='skyblue', ax=axes[0])
axes[0].set_title('Linear Scale', fontsize=13)
axes[0].set_xlabel('Word Count', fontsize=11)
axes[0].set_ylabel('Frequency (Number of Texts)', fontsize=11)

# Graph 2: Distribution in Logarithmic Scale
sns.histplot(train_dataset['word_count'], bins=100, color='skyblue', ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Logarithmic Scale', fontsize=13)
axes[1].set_xlabel('Word Count', fontsize=11)
axes[1].set_ylabel('Frequency (Log Scale)', fontsize=11)


plt.suptitle('Distribution of Lyrics Word Count: Linear vs Logarithmic Scale', fontsize=14, fontweight='bold')


plt.tight_layout()
plt.show()

**Observations:**

The visualization clearly demonstrates the **necessity of outlier removal**. In the linear scale plot, the standard song lengths are heavily compressed into a single spike, "hiding" the outliers. On the other hand, the logarithmic scale shows a long tail of lyrics reaching values up to 17,500 words. 

To ensure the integrity and quality of our model, we need to establish a maximum word count **threshold** to filter out these outliers. We will test a few different threshold values in the next steps to evaluate their impact on the dataset and determine the most optimal limit.

##### **3.2.2 Outliers Removal (Threshold)**


In [ ]:
# our threshold
threshold = 1400

# isolate the outliers based on the threshold
outliers = train_dataset[train_dataset['word_count'] > threshold]

print(f"Number of texts identified as extreme outliers (> {threshold} words): {len(outliers)}\n")

# observe the first 4 to confirm they are not songs or lyrics

# MUDAR Q  ISTO É CHAT





for i in range(min(4, len(outliers))):
    text_id = outliers.index[i]
    # print the first 500 characters of each outlier (it's enough to see the content)
    snippet = str(outliers.loc[text_id, 'raw_lyrics'])[:400] 
    print(f"--- OUTLIER ID: {text_id} (Words: {outliers.loc[text_id, 'word_count']}) ---")
    print(f"{snippet}...\n")

In [ ]:
train_dataset = train_dataset[train_dataset['word_count'] <= threshold]

print(f"Dataset size after applying the {threshold}-word threshold: {len(train_dataset)}")

In [ ]:
# our threshold
threshold = 15

# isolate the outliers based on the threshold
outliers = train_dataset[train_dataset['word_count'] < threshold]

print(f"Number of texts identified as extreme outliers (< {threshold} words): {len(outliers)}\n")

# observe the first 4 to confirm they are not songs or lyrics
for i in range(min(4, len(outliers))):
    text_id = outliers.index[i]
    # print the first 500 characters of each outlier (it's enough to see the content)
    snippet = str(outliers.loc[text_id, 'raw_lyrics'])[:400] 
    print(f"--- OUTLIER ID: {text_id} (Words: {outliers.loc[text_id, 'word_count']}) ---")
    print(f"{snippet}...\n")

In [ ]:
train_dataset = train_dataset[train_dataset['word_count'] >= threshold]

print(f"Dataset size after applying the {threshold}-word threshold: {len(train_dataset)}")

#### **3.2. Exploring the Lyrics Structure**

While loading the initial data, we noticed that the `raw_lyrics` column contains various elements such as structural tags (e.g., `[Verse 1]`, `(Chorus)`), standalone structural words and some formatting noise. Since these elements aren't important and will negatively impact the performance of our future NLP models, they need to be remove.

Therefore, this section aims to **identify and remove** this noise early in the data cleaning phase to ensure high-quality input

We will begin by isolating specific examples of the content that needs to be cleaned:

##### **3.2.1. Brackets (Squared or Regular ones)**

In [ ]:
regex_brackets = r'\[.*?\]|\(.*?\)'  # Matches text within brackets or parentheses

ex1_brackets = train_dataset[train_dataset['raw_lyrics'].str.contains(regex_brackets, na=False)].index[:4]  # Get the first 4 examples
ex1_brackets = train_dataset['raw_lyrics'].loc[ex1_brackets]
print("--- Example 1: Brackets ---")
print(ex1_brackets)

In [ ]:
train_dataset['cleaned_lyrics'] = train_dataset['raw_lyrics'].str.replace(regex_brackets, '', regex=True)
print("\n--- Example 1: Brackets Cleaned ---")
print(train_dataset['cleaned_lyrics'].loc[ex1_brackets.index])


##### **3.2.2. Standalone Structural Words**

We noticed songs that have those structural words (like 'Chorus' or 'Verse') but without the brackets. Which means they weren't removed before, but as we are not 100% sure about the exact words, we decided to extract the most frequent terms that appeared inside the brackets in our original column(`raw_lyrics`) because it is highy likely that the same words appear without any tags ( e.g. 'Chorus:' isntead of '[Chorus]').

In [ ]:
extracted_tags = train_dataset['raw_lyrics'].str.extractall(r'\[(.*?)\]')[0].str.lower()

core_words = extracted_tags.str.extract(r'([a-z]+)')[0] #turns "verse 1" into "verse"

top_words = core_words.value_counts().head(50)


print("--- FinalStructural Words (Filtered) ---")
print(top_words)

In [ ]:
regex_words_pattern = r'(?im)^\s*(?:' + '|'.join(top_words.index) + r')\b\s*\d*:?\s*$'

ex2_structural_words = train_dataset[train_dataset['cleaned_lyrics'].str.contains(regex_words_pattern, na=False)].index[3:6]
ex2_structural_words = train_dataset['cleaned_lyrics'].loc[ex2_structural_words]
print("--- Example 2: Structural Words ---")    
print(ex2_structural_words)

In [ ]:
train_dataset['cleaned_lyrics'] = train_dataset['cleaned_lyrics'].str.replace(
    regex_words_pattern, 
    '', 
    flags=re.MULTILINE, 
    regex=True
)

print("\n--- Example 2: Structural Words Cleaned ---")
print(train_dataset['cleaned_lyrics'].loc[ex2_structural_words.index])

### VER ESTE MARKDOWN

**A Note on Precision vs. Recall in Data Cleaning**

While our strict line-bound regex (`^...$`) successfully removes the vast majority of standalone structural tags (e.g., "Chorus:", "Intro"), it intentionally bypasses complex or heavily annotated tags (e.g., "First Verse:", "Part 1 - Tape Blip"). 

We chose to accept this limitation to prioritize data integrity. Attempting to capture every irregular tag format via hardcoded rules significantly increases the risk of false positives—inadvertently deleting genuine song lyrics. The remaining noise is statistically negligible and will not heavily impact our modeling phase.

##### **3.2.3. New Line Characters**

In [ ]:
regex_newlines = r'\n'  # Matches newline characters

ex3_newlines = train_dataset[train_dataset['cleaned_lyrics'].str.contains(regex_newlines, na=False)].index[:4]  # Get the first 4 examples
ex3_newlines = train_dataset['cleaned_lyrics'].loc[ex3_newlines]
print("--- Example 3: Newlines ---")
print(ex3_newlines)

In [ ]:
train_dataset['cleaned_lyrics'] = train_dataset['cleaned_lyrics'].str.replace(regex_newlines, ' ', regex=True)

print("\n--- Example 3: Newlines Cleaned ---")
print(train_dataset['cleaned_lyrics'].loc[ex3_newlines.index])

##### **3.2.4. Emojis**

In [ ]:
regex_emojis = r'[\u2600-\u27FF]|[\U00010000-\U0010ffff]'

ex4_emojis = train_dataset['cleaned_lyrics'].str.contains(regex_emojis, regex=True, na=False)
ex4_emojis_head3 = train_dataset[ex4_emojis].head(3).index

print("--- Example 4: Emojis ---")
print(train_dataset['cleaned_lyrics'].loc[ex4_emojis_head3])


print(f'\n There are {ex4_emojis.sum()} songs with emojis in the lyrics.')

**Decision**

While some rows containing emojis are valid lyrics, our analysis revealed that emojis often act as a strong indicator of non-musical text. Given that only 113 rows show this behavior, which is a statistically insignificant portion of our dataset, we chose to drop these rows entirely. 

In [ ]:
train_dataset = train_dataset[~ex4_emojis].reset_index(drop=True)

print(f"\nNew dataset size after dropping rows: {len(train_dataset)}")

##### **3.2.5. Mentions and URL's**

In [ ]:
regex_mentions_urls = r'@[A-Za-z0-9_]+|https?://\S+|www\.\S+'

ex5_mentions_urls = train_dataset['cleaned_lyrics'].str.contains(regex_mentions_urls, regex=True, na=False)
ex5_head7 = train_dataset[ex5_mentions_urls].head(7).index

print("--- Example 5: Mentions & URLs ---")
print(train_dataset['cleaned_lyrics'].loc[ex5_head7])

print(f"\nTotal rows with mentions or URLs: {ex5_mentions_urls.sum()}")

**Decision**

Similarly to the emoji, the presence of user mentions and specially hyperlinks strongly correlates with non-music with an even higher correlation than the emojis one. These are typically promotional spam, user comments etc.. Since there are only 217 rows (again a statistically insignificant portion of the dataset), we will drop these rows to preserve the quality of our model.

In [ ]:
train_dataset = train_dataset[~ex5_mentions_urls].reset_index(drop=True)

print(f"\nNew dataset size after dropping rows: {len(train_dataset)}")

##### **3.2.6. Dates (Journal Entries)**

In [ ]:
# Matches variations like: 12-05-22, 2022/05/12, 1-2-2000, 01/05/99
regex_dates = (r'^\b(?:'
    r'(?:0?[1-9]|[12]\d|3[01])[-](?:0?[1-9]|1[0-2])[-](?:19|20)?\d{2}|' # DD-MM-YY or DD-MM-YYYY
    r'(?:0?[1-9]|1[0-2])[-](?:0?[1-9]|[12]\d|3[01])[-](?:19|20)?\d{2}|' # MM-DD-YY or MM-DD-YYYY
    r'(?:19|20)\d{2}[-](?:0?[1-9]|1[0-2])[-](?:0?[1-9]|[12]\d|3[01])'   # YYYY-MM-DD
    r')\b')

ex6_dates = train_dataset['cleaned_lyrics'].str.contains(regex_dates, regex=True, na=False)
print(f"Rows with likely diary entries: {ex6_dates.sum()}")


print("\n--- Example 6: Diary Entries / ---")
print(train_dataset['cleaned_lyrics'].loc[ex6_dates])

In [ ]:
train_dataset = train_dataset[~ex6_dates].reset_index(drop=True)
print(f"Dataset size after cleaning: {len(train_dataset)}")

**Decision**
The decision to drop all this 13 entries was very straight forward as we're 100% sure that all of them journal entries with zero importance to our model

In [ ]:
#how do i localize the row with id 294 to check if it was removed?
train_dataset.loc[235547]


In [ ]:
def clean_lyrics_pipeline(text):